# RazorStrike v2 - clean-base multi-domain QLoRA (Colab Pro+ A100)

Base: `Qwen/Qwen3.6-35B-A3B` (clean, no merge, no abliteration).
Uncensoring is done via the `uncensor` data family, not weight ablation.

Flow: install -> auth -> clone -> **preflight (fail-fast)** -> build+push dataset -> train QLoRA -> (high-RAM) merge+push.
Runtime: **A100 (40GB)** for training. The merge step needs a **high-RAM** runtime (~70GB), not the GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
# Expect an A100 with ~40GB. A T4 (16GB) CANNOT load the 35B in 4-bit.

In [ ]:
!pip -q install -U "transformers>=4.57" "peft>=0.14" "bitsandbytes>=0.44" \
    "accelerate>=1.0" "datasets>=3.0" huggingface_hub

In [ ]:
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    os.environ.setdefault('HF_TOKEN', '')  # or paste here
login(os.environ['HF_TOKEN'])

In [ ]:
%cd /content
![ -d razorstrike ] || git clone https://github.com/lancejames221b/razorstrike.git
%cd /content/razorstrike
!git pull --ff-only

## 0. PREFLIGHT (fail-fast) - run this BEFORE the dataset build
Validates the riskiest untested step: 4-bit load of the custom multimodal
`Qwen3_5MoeForConditionalGeneration` (bitsandbytes on the SSM `in_proj_*` layers +
VLM loader) and that the LoRA targets actually match. ~5 min. It frees VRAM at the
end so the training cell has a clean 40GB. If it prints `PREFLIGHT OK`, the full run
is safe. If VRAM does not release, **restart the runtime** before training.

In [ ]:
import os, torch, gc
from transformers import AutoModelForImageTextToText, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
BASE='Qwen/Qwen3.6-35B-A3B'
bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
kw=dict(quantization_config=bnb, device_map={'':0}, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
try:
    m=AutoModelForImageTextToText.from_pretrained(BASE,**kw)
except Exception as e:
    print('ImageTextToText failed:',type(e).__name__,e); m=AutoModelForCausalLM.from_pretrained(BASE,**kw)
m=prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)
targets=['q_proj','k_proj','v_proj','o_proj','in_proj_qkv','in_proj_z','in_proj_a','in_proj_b','out_proj']
m=get_peft_model(m, LoraConfig(r=32,lora_alpha=64,lora_dropout=0.05,bias='none',
    task_type='CAUSAL_LM', target_modules=targets))
tp=sum(p.numel() for p in m.parameters() if p.requires_grad); tot=sum(p.numel() for p in m.parameters())
pct=100*tp/tot; print(f'trainable {tp:,} ({pct:.4f}% of {tot:,})')
assert pct>0.01, 'LoRA targets did not match - fix target_modules before training'
before=torch.cuda.memory_allocated()/1e9
del m
gc.collect(); gc.collect(); torch.cuda.empty_cache()
after=torch.cuda.memory_allocated()/1e9
print(f'VRAM {before:.1f}GB -> {after:.1f}GB after cleanup (freed {before-after:.1f}GB)')
assert after < 3.0, f'VRAM not released ({after:.1f}GB resident) - RESTART RUNTIME before training to avoid OOM'
print('PREFLIGHT OK - 4bit VLM load + LoRA attach works, VRAM freed; safe to build dataset + train')

## 1. Build + push the combined dataset (all 8 families)
Uncapped: real downloads (decompile-bench->40k, NuminaMath 40k, cyber ~30k, mythos ~8k, plus synthetic families). Pushes to `DATA_REPO`.

In [ ]:
import os
os.environ['DATA_REPO'] = 'lancejames221b/razorstrike-v2-sft'
os.environ['PUSH'] = '1'
!python -m scripts.build_dataset

## 2. Train the text QLoRA
4-bit base fits the 40GB A100. Watch the `[guard] trainable params` line - it must be a non-trivial %.

In [ ]:
import os
os.environ['BASE_REPO']    = 'Qwen/Qwen3.6-35B-A3B'
os.environ['DATA_REPO']    = 'lancejames221b/razorstrike-v2-sft'
os.environ['ADAPTER_REPO'] = 'lancejames221b/razorstrike-v2-offsec-lora'
os.environ['OUT_DIR']      = '/content/adapter'
os.environ['MAXLEN']       = '4096'
os.environ['TARGET_MLP']   = '0'   # 1 = also adapt the 256 MoE experts (large)
# os.environ['RESUME'] = '1'       # resume from checkpoint after a disconnect
!python -m scripts.train_lora

## 3. Merge + publish (HIGH-RAM runtime, not the GPU)
35B bf16 merge needs ~70GB RAM. Switch to a high-RAM runtime; the adapter is safe on HF from step 2. Skippable if you serve base+adapter.

In [ ]:
import os
os.environ['BASE_REPO']    = 'Qwen/Qwen3.6-35B-A3B'
os.environ['ADAPTER_DIR']  = '/content/adapter'
os.environ['MERGED_REPO']  = 'lancejames221b/razorstrike-v2'
!python -m scripts.merge_push